# Principled Attribution: Shapley Values

Game-theoretic attribution methods that satisfy fundamental axioms
(efficiency, symmetry, null player). Unlike gradient-based methods,
Shapley values give faithfully additive importance scores.

This notebook covers the `irtk.principled_attribution` module.

## Why This Matters

Shapley values and KernelSHAP provide game-theoretically principled attribution scores that satisfy desirable axioms (efficiency, symmetry, null player). While more expensive than gradient-based methods, they give unambiguous answers about component importance and can detect interaction effects.

**Key references:**
- [Sundararajan & Najmi (2020) "The Many Shapley Values"](https://arxiv.org/abs/1908.08474) — Principled attribution using game-theoretic methods

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import principled_attribution

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Shapley Value Token Attribution

Which tokens most contribute to the prediction?

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")
paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

result = principled_attribution.shapley_value_tokens(
    model, tokens, metric, n_samples=100
)

token_strs = [tokenizer.decode([int(t)]) for t in tokens]
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['green' if v > 0 else 'red' for v in result['shapley_values']]
ax.bar(range(len(tokens)), result['shapley_values'], color=colors)
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(token_strs, rotation=45, ha='right')
ax.set_ylabel('Shapley Value')
ax.set_title('Token Attribution (Shapley Values) for "Paris"')
plt.tight_layout()
plt.show()

print(f"Efficiency gap: {result['efficiency_gap']:.4f} (should be ~0)")
print(f"Full value: {result['full_value']:.4f}, Null value: {result['null_value']:.4f}")

## 2. Component Interaction Index

Do model components have synergistic or redundant effects?

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")
components = [f"blocks.{i}.hook_attn_out" for i in range(model.cfg.n_layers)]

result = principled_attribution.interaction_index(model, tokens, metric, components)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(result['interaction_matrix'], cmap='RdBu_r', aspect='auto')
ax.set_xlabel('Attention Layer')
ax.set_ylabel('Attention Layer')
ax.set_title('Attention Layer Interaction (+ = synergistic)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print(f"Most synergistic: {result['most_synergistic']}")
print(f"Most redundant: {result['most_redundant']}")

## 3. Path Attribution

Layer-by-layer decomposition of computation.

In [ ]:
result = principled_attribution.path_attribution_value(model, tokens, metric)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(model.cfg.n_layers)
ax.bar(x - 0.2, result['attn_attributions'], 0.4, label='Attention', color='steelblue')
ax.bar(x + 0.2, result['mlp_attributions'], 0.4, label='MLP', color='coral')
ax.set_xlabel('Layer')
ax.set_ylabel('Attribution')
ax.set_title('Path Attribution: Attention vs MLP')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Residual fraction: {result['residual_fraction']:.4f}")

## 4. Importance Ranking with Confidence

Rank components with statistical significance across prompts.

In [ ]:
prompts = [
    "The Eiffel Tower is located in",
    "The capital of France is",
    "Paris is the capital of",
]
token_seqs = [model.to_tokens(p) for p in prompts]
components = ([f"blocks.{i}.hook_attn_out" for i in range(model.cfg.n_layers)] +
              [f"blocks.{i}.hook_mlp_out" for i in range(model.cfg.n_layers)])

result = principled_attribution.importance_ranking_with_std(
    model, token_seqs, metric, components
)

print("Component ranking (mean +/- std):")
for name, mean, std in result['ranking'][:10]:
    sig = " *" if name in result['significant_components'] else ""
    print(f"  {name:>30s}: {mean:+.4f} +/- {std:.4f}{sig}")
print(f"\n* = statistically significant (|mean| > 2*std)")

## Summary

| Function | Purpose |
|----------|--------|
| `shapley_value_tokens()` | Shapley value importance for input tokens |
| `kernel_shap_activations()` | Fast SHAP for activation dimensions |
| `interaction_index()` | Pairwise synergy/redundancy between components |
| `path_attribution_value()` | Layer-by-layer attribution decomposition |
| `importance_ranking_with_std()` | Component ranking with confidence intervals |